# Chapter 14 -- FT-Transformer

Reproduces:
- Figure 14.1: per-layer symmetric energy fraction $\alpha_S$ and skew-symmetric
  energy fraction $\alpha_A$ of an FT-Transformer trained on two small tabular
  classification datasets (breast_cancer and iris).
- Table 14.1: per-layer $\alpha_A$ values and post-hoc projection accuracies
  (full / sym-only / asym-only).

Runtime target: < 15 min on a single GPU (CLAUDE.md default environment).


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tabkernels.architectures.ft_transformer import FTTransformer
from tabkernels.transparency import inspect_attention

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Resolve the figure directory relative to this notebook's location.
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIG_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('figure dir:', FIG_DIR)


## Helpers: a tiny supervised trainer

FT-Transformer is a feed-forward classifier; we train it the standard way.


In [ ]:
def fit_ft_transformer(
    X_train, y_train, X_test, y_test, n_classes,
    *, n_epochs=80, lr=3e-3, weight_decay=1e-5, seed=0,
    d_model=64, n_heads=8, n_layers=3, dim_ff=128, dropout=0.1,
):
    """Train an FTTransformer on (X_train, y_train); return (model, val_acc).

    All tensors are moved to ``device`` (GPU when available).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    n_features = X_train.shape[1]
    Xtr = torch.as_tensor(X_train, dtype=torch.float32, device=device)
    ytr = torch.as_tensor(y_train, dtype=torch.long, device=device)
    Xte = torch.as_tensor(X_test, dtype=torch.float32, device=device)
    yte = torch.as_tensor(y_test, dtype=torch.long, device=device)
    model = FTTransformer(
        n_num_features=n_features,
        cat_cardinalities=[],
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        dim_ff=dim_ff,
        n_classes=n_classes,
        dropout=dropout,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    bs = 64
    n = Xtr.shape[0]
    for _ in range(n_epochs):
        perm = torch.randperm(n, device=device)
        model.train()
        for i in range(0, n, bs):
            idx = perm[i : i + bs]
            opt.zero_grad()
            logits = model(Xtr[idx])
            loss = loss_fn(logits, ytr[idx])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(Xte).argmax(dim=-1)
        val_acc = float((preds == yte).float().mean().item())
    return model, val_acc, (Xte, yte)


## Train on two small tabular datasets

We use ``sklearn`` datasets so the notebook stays offline and small. The two
tasks have different feature counts and label cardinalities; both are
well-studied tabular benchmarks.


In [ ]:
datasets = {}
for name, loader, n_classes in [
    ('breast_cancer', load_breast_cancer, 2),
    ('iris', load_iris, 3),
]:
    data = loader()
    X = data.data.astype(np.float32)
    y = data.target.astype(np.int64)
    X = StandardScaler().fit_transform(X)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=0, stratify=y
    )
    datasets[name] = dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, n_classes=n_classes)

models = {}
for name, ds in datasets.items():
    model, val_acc, (Xte, yte) = fit_ft_transformer(
        ds['Xtr'], ds['ytr'], ds['Xte'], ds['yte'], ds['n_classes'],
        n_epochs=80, lr=3e-3, d_model=64, n_heads=8, n_layers=3,
        dim_ff=128, dropout=0.1, seed=0,
    )
    models[name] = dict(model=model, val_acc=val_acc, Xte=Xte, yte=yte)
    print(f'{name:>14s}: val_acc = {val_acc:.3f}')


## Apply the Chapter 13 decomposition lens

We call ``tabkernels.transparency.inspect_attention`` on each trained model,
then aggregate the per-head energy fractions per layer. Because the lens
recognises the inner ``StdAttentionDecomposable`` modules of every encoder
layer, it returns a $\bigl(L \times H\bigr)$ list of $\alpha_A$ values.


In [ ]:
def per_layer_alpha(model, Xte, yte):
    """Return per-layer mean alpha_S, alpha_A and the inspect_attention report."""
    model.eval()
    blocks = model.attention_blocks()
    per_layer = []
    for block in blocks:
        a_s, a_a = block.energy_split()
        per_layer.append({'alpha_S': a_s, 'alpha_A': a_a})
    # Run the full inspection workflow for the post-hoc projection metrics.
    report = inspect_attention(model, Xte, y=yte)
    return per_layer, report

results = {}
for name, info in models.items():
    per_layer, report = per_layer_alpha(info['model'], info['Xte'], info['yte'])
    results[name] = dict(per_layer=per_layer, report=report)
    print(f'\n=== {name} ===')
    print(f'  val_acc                 : {info["val_acc"]:.3f}')
    print(f'  full / sym / asym acc   : '
          f'{report["full_acc"]:.3f} / {report["sym_only_acc"]:.3f} / '
          f'{report["asym_only_acc"]:.3f}')
    print(f'  head-mean alpha_S/alpha_A: {report["alpha_S"]:.3f} / '
          f'{report["alpha_A"]:.3f}')
    for L, d in enumerate(per_layer):
        print(f'  layer {L}: alpha_S={d["alpha_S"]:.3f}, alpha_A={d["alpha_A"]:.3f}')


## Figure 14.1: per-layer $\alpha_A$ across datasets


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.0, 3.5))
colors = {'breast_cancer': 'C0', 'iris': 'C1'}
markers = {'breast_cancer': 'o', 'iris': 's'}
for name, info in results.items():
    alphas = [d['alpha_A'] for d in info['per_layer']]
    ax.plot(
        range(len(alphas)), alphas,
        marker=markers[name], color=colors[name], lw=1.4,
        label=name.replace('_', ' '),
    )
ax.axhline(0.5, color='gray', lw=0.6, ls='--', label=r'symmetric break-even')
ax.set_xlabel('encoder layer index')
ax.set_ylabel(r'$\alpha_A$ (skew-symmetric energy fraction)')
ax.set_title(r'FT-Transformer per-layer $\alpha_A$')
ax.set_ylim(0.0, 1.0)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'fig_14_01_alpha_per_layer.pdf')
fig.savefig(fig_path, bbox_inches='tight')
print('saved', fig_path)
plt.show()


## Table 14.1: per-layer $\alpha_A$ (LaTeX-printable)


In [ ]:
names = list(results.keys())
n_layers = max(len(results[n]['per_layer']) for n in names)
rows = []
rows.append('Layer  ' + '  '.join(f'{n:>14s}' for n in names))
for L in range(n_layers):
    cells = []
    for n in names:
        v = results[n]['per_layer'][L]['alpha_A']
        cells.append(f'{v:>14.3f}')
    rows.append(f'{L:5d}  ' + '  '.join(cells))
row_avg = '  '.join(
    f'{np.mean([d["alpha_A"] for d in results[n]["per_layer"]]):>14.3f}'
    for n in names
)
rows.append(f' mean  ' + '  '.join(row_avg.split()))
print('\n'.join(rows))

# LaTeX form -- copy/paste-ready for the chapter.
print('\n% --- LaTeX (Table 14.1) ---')
print(r'\begin{tabular}{r' + 'r' * len(names) + '}')
print(r'  \toprule')
header = r'  Layer & ' + ' & '.join(
    n.replace('_', r'\_') for n in names
) + r' \\'
print(header)
print(r'  \midrule')
for L in range(n_layers):
    line = f'  {L} & '
    line += ' & '.join(
        f'{results[n]["per_layer"][L]["alpha_A"]:.3f}' for n in names
    )
    line += r' \\'
    print(line)
print(r'  \bottomrule')
print(r'\end{tabular}')


## Reading

On both small i.i.d. classification tasks the per-layer $\alpha_A$ stays close
to $0.5$ -- not because the asymmetric component is doing predictive work, but
because at modest training-data scales the bilinear form $B = W_Q^\top W_K$ has
not committed enough energy in either direction to deviate from a generic
(roughly equal-energy) initialisation. The post-hoc projection accuracies
(``sym_only_acc`` versus ``full_acc``) are the load-bearing diagnostic: when
they agree, the asymmetric capacity is not pulling its weight, exactly the
situation Chapter 13 calls out. Chapter 18 returns to this comparison across
FT-Transformer, SAINT, TabPFN, and TabICL.
